In [3]:
import numpy as np
import tensorflow as tf
import time
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

# ===========================
# STEP 1: DATASET
# ===========================
print("\n" + "="*70)
print("STEP 1: DATASET PREPARATION")
print("="*70)

data = """Natural language processing is a field of artificial intelligence
Natural language models are used in NLP
RNN is used for sequence modeling in NLP
Language models predict the next word in a sentence
Deep learning models like RNN and LSTM are great for text generation
Pimpri Chinchwad College of Engineering provides quality education"""

print("Dataset Loaded Successfully!")

# ===========================
# STEP 2: TOKENIZATION
# ===========================
print("\n" + "="*70)
print("STEP 2: TOKENIZATION")
print("="*70)

tokenizer = Tokenizer()
tokenizer.fit_on_texts([data])

total_words = len(tokenizer.word_index) + 1
print("Total Words:", total_words)

# ===========================
# STEP 3: SEQUENCE CREATION
# ===========================
print("\n" + "="*70)
print("STEP 3: SEQUENCE CREATION")
print("="*70)

input_sequences = []

for line in data.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i+1])

max_len = max(len(x) for x in input_sequences)

input_sequences = np.array(
    pad_sequences(input_sequences, maxlen=max_len, padding='pre')
)

X, y = input_sequences[:, :-1], input_sequences[:, -1]
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

print("Total Sequences:", len(X))
print("Max Sequence Length:", max_len)

# ===========================
# STEP 4: MODEL
# ===========================
print("\n" + "="*70)
print("STEP 4: BUILDING LSTM MODEL")
print("="*70)

model = Sequential([
    Embedding(total_words, 50),
    LSTM(150),
    Dense(total_words, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])


model.build(input_shape=(None, max_len-1))

print("\nMODEL SUMMARY:")
model.summary()

# ===========================
# STEP 5: TRAINING
# ===========================
print("\n" + "="*70)
print(f"STEP 5: TRAINING MODEL - {time.strftime('%H:%M:%S')}")
print("="*70)

history = model.fit(X, y, epochs=300, verbose=1)

# ===========================
# TEXT GENERATION FUNCTION
# ===========================
def generate_text(seed_text, next_words, model, max_len):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_len-1, padding='pre')

        predicted = np.argmax(model.predict(token_list, verbose=0), axis=-1)[0]

        for word, index in tokenizer.word_index.items():
            if index == predicted:
                seed_text += " " + word
                break
    return seed_text

# ===========================
# STEP 6: TEXT GENERATION
# ===========================
print("\n" + "="*70)
print("STEP 6: TEXT GENERATION")
print("="*70)

seeds = [
    "Natural language",
    "Deep learning",
    "Pimpri Chinchwad"
]

for s in seeds:
    print(f"\nSeed: {s}")
    print("Generated:", generate_text(s, 6, model, max_len))

# ===========================
# FINAL RESULT
# ===========================
print("\n" + "="*70)
print("FINAL RESULT")
print("="*70)

seed = "Natural language"
output = generate_text(seed, 6, model, max_len)

print("Seed Input:", seed)
print("Generated Output:", output)


STEP 1: DATASET PREPARATION
Dataset Loaded Successfully!

STEP 2: TOKENIZATION
Total Words: 39

STEP 3: SEQUENCE CREATION
Total Sequences: 47
Max Sequence Length: 12

STEP 4: BUILDING LSTM MODEL

MODEL SUMMARY:


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 11, 50)         │         1,950 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 150)            │       120,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 39)             │         5,889 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 128,439 (501.71 KB)

 Trainable params: 128,439 (501.71 KB)

 Non-trainable params: 0 (0.00 B)


STEP 5: TRAINING MODEL - 06:38:09
Epoch 1/300
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 50ms/step - accuracy: 0.0426 - loss: 3.6648
Epoch 2/300
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - accuracy: 0.0851 - loss: 3.6552
Epoch 3/300
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.0851 - loss: 3.6473
Epoch 4/300
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.0638 - loss: 3.6375
Epoch 5/300
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.0638 - loss: 3.6255
Epoch 6/300
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.0638 - loss: 3.6100
Epoch 7/300
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.0638 - loss: 3.5829
Epoch 8/300
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.0638 - loss: 3.5454
Epoch 9/300
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.0638 - loss: 3.4992
Epoch 10/300
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.0638 - loss: 3.4512
Epoch 11/300
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.0638 - loss: 3.4400 
Epoch 12/300
2/2 ━━━━━━━━━━━━━━━━━━━━ 